# SynFlow Top-5: L2 Evaluation (TPU, 50 epochs)

Fair comparison: SynFlow top-5 vs HBO top-5 vs Random top-5

SynFlow top-5 (from local CPU computation):
1. W=96 D=6 BN NoSkip  (score=3.80e+02)
2. W=96 D=5 BN NoSkip  (score=3.58e+02)
3. W=96 D=3 BN NoSkip  (score=3.38e+02)
4. W=64 D=6 BN NoSkip  (score=3.03e+02)
5. W=64 D=5 BN NoSkip  (score=2.79e+02)

In [ ]:
import sys
import time
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms

# TPU setup
import torch_xla.core.xla_model as xm
import torch_xla.distributed.xla_multiprocessing as xmp
import torch_xla.distributed.parallel_loader as pl

DEVICE = xm.xla_device()
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Data loading — same protocol as HBO_revised notebook
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

full_train = torchvision.datasets.CIFAR10('/tmp/data', train=True, download=True, transform=transform_train)
test_ds = torchvision.datasets.CIFAR10('/tmp/data', train=False, download=True, transform=transform_test)

# Split into train/val (same seed as HBO_revised)
np.random.seed(42)
indices = np.random.permutation(len(full_train))
val_size = 5000
val_idx, train_idx = indices[:val_size], indices[val_size:]
train_ds = Subset(full_train, train_idx)
val_ds = Subset(full_train, val_idx)

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

In [ ]:
# SynFlow top-5 configs (from local CPU computation, same search space as HBO)
# These are the highest-SynFlow-score architectures in the search space
SYNFLOW_TOP5 = [
    {'width': 96, 'depth': 6, 'norm': 'batchnorm', 'skip': False},  # rank 1  score=3.80e+02
    {'width': 96, 'depth': 5, 'norm': 'batchnorm', 'skip': False},  # rank 2  score=3.58e+02
    {'width': 96, 'depth': 3, 'norm': 'batchnorm', 'skip': False},  # rank 3  score=3.38e+02
    {'width': 64, 'depth': 6, 'norm': 'batchnorm', 'skip': False},  # rank 4  score=3.03e+02
    {'width': 64, 'depth': 5, 'norm': 'batchnorm', 'skip': False},  # rank 5  score=2.79e+02
]

EPOCHS_L1 = 10   # L1 screening
EPOCHS_L2 = 50   # L2 final evaluation
LR = 0.05
SEEDS = [42, 123]
CHECKPOINT_EVERY = 10
LOG_EVERY = 5

In [ ]:
def build_model(width, depth, norm_type, skip=False):
    """Plain ConvNet matching HBO_revised architecture definition."""
    layers = []
    c = 3
    for i in range(depth):
        layers.append(nn.Conv2d(c, width, 3, padding=1))
        if norm_type == 'batchnorm':
            layers.append(nn.BatchNorm2d(width))
        elif norm_type == 'layernorm':
            layers.append(nn.LayerNorm([width, 32, 32]))
        c = width
    if skip:
        layers.append(nn.Identity())  # placeholder for skip
    layers.extend([
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(c, 10)
    ])
    return nn.Sequential(*layers)

In [ ]:
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(x)
        loss = F.cross_entropy(out, y)
        loss.backward()
        xm.optimizer_step(optimizer)
        total_loss += loss.item() * x.size(0)
    scheduler.step()
    return total_loss / len(loader.dataset)


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = F.cross_entropy(out, y)
            total_loss += loss.item() * x.size(0)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return total_loss / total, correct / total

In [ ]:
def train_model(cfg, epochs, seed=42, log_every=5):
    """Train one config, return full epoch-by-epoch history."""
    torch.manual_seed(seed)
    model = build_model(cfg['width'], cfg['depth'], cfg['norm'], cfg['skip'])
    model = model.to(DEVICE)
    
    optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    history = []
    best_val_loss = float('inf')
    best_val_acc = 0.0
    start = time.time()
    
    for ep in range(epochs):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler)
        val_loss, val_acc = evaluate(model, val_loader)
        
        # Track best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
        
        # Log every N epochs
        if (ep + 1) % log_every == 0 or ep == 0:
            elapsed = time.time() - start
            print(f"    ep={ep+1:3d}/{epochs}  train={train_loss:.4f}  val={val_loss:.4f}  acc={val_acc:.4f}  ({elapsed:.0f}s)", flush=True)
        
        # Checkpoint
        if (ep + 1) % 10 == 0:
            xm.save(model.state_dict(), f"/tmp/ckpt_W{cfg['width']}_D{cfg['depth']}_ep{ep+1}_seed{seed}.pt")
        
        history.append({'epoch': ep+1, 'train_loss': train_loss, 'val_loss': val_loss, 'val_acc': val_acc})
    
    test_loss, test_acc = evaluate(model, test_loader)
    
    return {
        'history': history,
        'best_val_loss': best_val_loss,
        'best_val_acc': best_val_acc,
        'test_loss': test_loss,
        'test_acc': test_acc
    }

In [ ]:
# ── Phase 1: L1 (10 epochs) ─────────────────────────────────────────────
print("=" * 70)
print("Phase 1: L1 (10 epochs) — SynFlow top-5")
print("=" * 70)

sys.path.insert(0, '/home/node/.openclaw/workspace/github_staging/ThermoRG-NN')
from thermorg.topology_calculator import compute_J_topo

l1_results = []
for i, cfg in enumerate(SYNFLOW_TOP5):
    cfg_label = f"W={cfg['width']} D={cfg['depth']} {cfg['norm']} skip={cfg['skip']}"
    print(f"\n[{i+1}/5] {cfg_label}")
    
    # Compute J_topo
    model_for_J = build_model(cfg['width'], cfg['depth'], cfg['norm'], cfg['skip'])
    J, _ = compute_J_topo(model_for_J)
    
    l1_losses = []
    for seed in SEEDS:
        t0 = time.time()
        result = train_model(cfg, EPOCHS_L1, seed, log_every=5)
        elapsed = time.time() - t0
        l1_losses.append(result['best_val_loss'])
        print(f"  seed={seed}: L1_best_val={result['best_val_loss']:.4f}  ({elapsed:.0f}s)")
    
    l1_results.append({
        'cfg': cfg,
        'J_topo': J,
        'l1_losses': l1_losses,
        'l1_mean': np.mean(l1_losses)
    })
    print(f"  → L1 mean: {np.mean(l1_losses):.4f}  J_topo={J:.4f}")

In [ ]:
# ── Phase 2: L2 (50 epochs) ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("Phase 2: L2 (50 epochs) — SynFlow top-5")
print("=" * 70)

l2_results = []
for i, cfg in enumerate(SYNFLOW_TOP5):
    cfg_label = f"W={cfg['width']} D={cfg['depth']} {cfg['norm']} skip={cfg['skip']}"
    print(f"\n[{i+1}/5] {cfg_label}")
    
    l2_losses = []
    l2_accs = []
    for seed in SEEDS:
        t0 = time.time()
        result = train_model(cfg, EPOCHS_L2, seed, log_every=10)
        elapsed = time.time() - t0
        l2_losses.append(result['best_val_loss'])
        l2_accs.append(result['test_acc'])
        print(f"  seed={seed}: L2_val={result['best_val_loss']:.4f}  test_acc={result['test_acc']:.4f}  ({elapsed:.0f}s)", flush=True)
    
    l2_mean = np.mean(l2_losses)
    l2_std = np.std(l2_losses)
    l2_acc_mean = np.mean(l2_accs)
    l2_results.append({
        'cfg': cfg,
        'l2_losses': l2_losses,
        'l2_mean': l2_mean,
        'l2_std': l2_std,
        'l2_acc_mean': l2_acc_mean
    })
    print(f"  → L2 mean: {l2_mean:.4f} ± {l2_std:.4f}  test_acc: {l2_acc_mean:.4f}")

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("FINAL RESULTS: SynFlow Top-5 — L2 (50 epochs)")
print("=" * 70)
print(f"{'Rank':<5} {'Config':<25} {'J_topo':>8} {'L2 Loss':>10} {'Test Acc':>10}")
print("-" * 70)

# Sort by L2 mean
sorted_results = sorted(l2_results, key=lambda x: x['l2_mean'])
for rank, r in enumerate(sorted_results, 1):
    cfg = r['cfg']
    label = f"W={cfg['width']} D={cfg['depth']} BN NS"
    # Get J_topo from L1 results (same cfg)
    j_topo = next(x['J_topo'] for x in l1_results if x['cfg'] == cfg)
    print(f"{rank:<5} {label:<25} {j_topo:>8.4f} {r['l2_mean']:>10.4f} {r['l2_acc_mean']:>10.4f}")

print("\n" + "-" * 70)
print("Comparison with HBO and Random (50-epoch L2 from HBO_revised run):")
print("-" * 70)
print(f"{'Method':<15} {'Best Config':<25} {'Best Loss':>10} {'Best Acc':>10}")
print("-" * 70)
print(f"{'SynFlow':<15} {'W=96 D=6 BN NoSkip':<25} {sorted_results[0]['l2_mean']:>10.4f} {sorted_results[0]['l2_acc_mean']:>10.4f}")
print(f"{'HBO':<15} {'W=96 D=6 BN NoSkip':<25} {0.3770:>10.4f} {0.8744:>10.4f}")
print(f"{'Random':<15} {'W=64 D=6 BN NoSkip':<25} {0.4270:>10.4f} {0.8515:>10.4f}")

In [ ]:
# Save results for figure generation
output = {
    'experiment': 'synflow_l2_tpu',
    'epochs_l1': EPOCHS_L1,
    'epochs_l2': EPOCHS_L2,
    'seeds': SEEDS,
    'synflow_top5': [
        {
            'cfg': r['cfg'],
            'J_topo': next(x['J_topo'] for x in l1_results if x['cfg'] == r['cfg']),
            'l1_mean': next(x['l1_mean'] for x in l1_results if x['cfg'] == r['cfg']),
            'l2_mean': r['l2_mean'],
            'l2_std': r['l2_std'],
            'l2_acc_mean': r['l2_acc_mean']
        }
        for r in l2_results
    ],
    'hbo_reference': {
        'best_loss': 0.3770,
        'best_acc': 0.8744,
        'best_config': 'W=96 D=6 BN NoSkip'
    },
    'random_reference': {
        'best_loss': 0.4270,
        'best_acc': 0.8515,
        'best_config': 'W=64 D=6 BN NoSkip'
    }
}

with open('/tmp/synflow_l2_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print("Results saved to /tmp/synflow_l2_results.json")

# Print JSON for direct copy
print("\nJSON for paper:\n")
print(json.dumps(output, indent=2))